In [ ]:
#Imports
import torch
import csv
from matplotlib import pyplot as plt
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader, TensorDataset

#Data loading
with open("mnist_train.csv", "r") as read_obj:
    csv_reader = csv.reader(read_obj)
    train = list(csv_reader)
train = train[1:]
train = [[int(train[i][j]) for j in range(len(train[0]))] for i in range(len(train))]
train = torch.tensor(train, dtype = torch.float32)

#Data formatting and data loader
train_Y = train[:,0].long()
train_X = train[:,1:]

dataset = TensorDataset(train_X, train_Y)

batch_size = 16
dataloader = DataLoader(dataset, batch_size = batch_size, shuffle = True)

In [ ]:
#Model declarations
#Encoder model
class Encoder(nn.Module):
    def __init__(self, latent_dim = 128):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(28*28, 512),
            nn.ReLU(),
            nn.Linear(512, 256),
            nn.ReLU(),
            nn.Linear(256, latent_dim)
        )

    def forward(self, x):
        return self.net(x)
    
#Predictor model
#TODO add extra variables for additional context for predictor model
class Predictor(nn.Module):
    def __init__(self, latent_dim = 128):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(latent_dim, latent_dim),
            nn.ReLU(),
            nn.Linear(latent_dim, latent_dim)
        )
    
    def forward(self, x):
        return self.net(x)

#JEPA wrapper module
class MNIST_JEPA(nn.Module):
    def __init__(self, latent_dim = 128, ema = 0.99):
        super().__init__()
        self.context_encoder = Encoder(latent_dim)
        self.target_encoder = Encoder(latent_dim)
        self.predictor = Predictor(latent_dim)
        self.ema = ema

        #update target encoders weights to be identical to context encoder
        self.update_target_encoder(momentum = 0)

        #target encoder will inherit context encoder weights
        #so no need to track gradients
        for param in self.target_encoder.parameters():
            param.requires_grad = False
        
    #updates target encoder weights
    @torch.no_grad()
    def update_target_encoder(self, momentum = None):
        m = self.ema if momentum is None else momentum
        for param_context, param_target in zip(self.context_encoder.parameters(), self.target_encoder.parameters()):
            param_target.data = m*param_target + (1-m)*param_context
    
    #TODO make a good masking function for already flattened vectors
    def apply_mask(self, x):
        context_x = x.clone()
        batch_size = x.shape[0]

        pass

    def forward(self, x):
        x_masked, action = self.apply_mask(x)
        x_target = x

        z_context = self.context_encoder(x_masked)

        with torch.no_grad():
            z_target = self.target_encoder(x_target)
        
        z_pred = self.predictor(z_context)

        return z_pred, z_target.detach()

